# Mediterranean 10-Metre Wind Speed — Climatology, Anomalies & Extremes (1979–2022)

**Workflow**
1. Open ERA5 10-metre U and V wind components from the `era5-pds` collection on Microsoft Planetary Computer — monthly means, lazy (dask).
2. Compute the **yearly mean wind speed** at every pixel and explore it with a year slider.
3. Compute the **long-term climatological mean** wind speed at every pixel (mean over all years).
4. Compute the **monthly wind speed anomaly** = monthly value − pixel climatological mean.
5. Visualise the monthly anomaly with an interactive time slider.
6. At every pixel, find the **minimum** and **maximum** of the anomaly time-series.
7. Plot spatial maps of the min-anomaly and max-anomaly fields.

> **Access**: ERA5-PDS on Planetary Computer is publicly accessible — no credentials needed.

In [ ]:
!pip install adlfs

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import xarray as xr
import dask
import holoviews as hv
import hvplot.xarray
import panel as pn
import pystac_client
import planetary_computer

pn.extension()
hv.config.image_rtol = 0.01

In [ ]:
# Mediterranean bounding box (ERA5 uses 0–360 longitude)
LAT_MIN, LAT_MAX     = 30.0, 47.0
LON_MIN_W, LON_MAX_W = 354.0, 360.0   # -6° to 0°E
LON_MIN_E, LON_MAX_E =   0.0,  42.0   #  0° to 42°E

# ERA5-PDS on Planetary Computer spans 1979-01 through 2022-12
TIME_START = "1979-01"
TIME_END   = "2022-12"

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

## 1 — Open ERA5 wind datasets (lazy)

Each month of ERA5-PDS is a separate STAC item.  We open every month's Zarr store lazily and concatenate along time.

In [ ]:
def open_era5_asset(signed_item, asset_key):
    """Open one ERA5-PDS Zarr asset as an xarray Dataset."""
    asset = signed_item.assets[asset_key]
    open_kwargs = asset.extra_fields["xarray:open_kwargs"].copy()
    storage_options = open_kwargs.pop("storage_options")
    open_kwargs.pop("engine", None)
    return xr.open_zarr(asset.href, storage_options=storage_options, **open_kwargs)


def subset_med(da):
    """Subset to Mediterranean and convert lon to -180..180."""
    da_lat = da.sel(lat=slice(LAT_MAX, LAT_MIN))
    west   = da_lat.sel(lon=slice(LON_MIN_W, LON_MAX_W))
    east   = da_lat.sel(lon=slice(LON_MIN_E, LON_MAX_E))
    da_med = xr.concat([west, east], dim="lon").sortby("lon")
    new_lon = ((da_med.lon.values + 180) % 360 - 180)
    return da_med.assign_coords(lon=new_lon).sortby("lon")

In [ ]:
%%time
# Search all analysis items in the full date range
search = catalog.search(
    collections=["era5-pds"],
    datetime=f"{TIME_START}/{TIME_END}",
    query={"era5:kind": {"eq": "an"}},
)
items = list(search.items())
items.sort(key=lambda i: i.id)
print(f"Found {len(items)} analysis items ({TIME_START} – {TIME_END})")

In [ ]:
%%time
u_list, v_list = [], []

for item in items:
    signed = planetary_computer.sign(item)
    ds_u = open_era5_asset(signed, "eastward_wind_at_10_metres")
    ds_v = open_era5_asset(signed, "northward_wind_at_10_metres")

    t = ds_u.time.values[0]

    # Subset spatially first (reduces data loaded), then compute monthly mean eagerly.
    # Computing inside the loop keeps the dask graph small; deferring 528 months
    # of hourly zarr reads into one graph exhausts the scheduler.
    u_mon = subset_med(ds_u["eastward_wind_at_10_metres"]).mean(dim="time").compute()
    v_mon = subset_med(ds_v["northward_wind_at_10_metres"]).mean(dim="time").compute()

    u_list.append(u_mon.expand_dims(time=[t]))
    v_list.append(v_mon.expand_dims(time=[t]))

u10 = xr.concat(u_list, dim="time")
v10 = xr.concat(v_list, dim="time")

# Snap timestamps to month-start (assign_coords is the correct xarray idiom;
# item-assignment `da["time"] = ...` does not update coordinates)
month_start = pd.to_datetime(u10.time.values).to_period("M").to_timestamp()
u10 = u10.assign_coords(time=month_start)
v10 = v10.assign_coords(time=month_start)

print(f"U10 shape: {dict(zip(u10.dims, u10.shape))}")
print(f"V10 shape: {dict(zip(v10.dims, v10.shape))}")
u10

In [ ]:
# Wind speed magnitude — still lazy
wspd = np.sqrt(u10**2 + v10**2)
wspd.attrs["long_name"] = "10-Metre Wind Speed"
wspd.attrs["units"]     = "m/s"
print(f"Wind speed shape: {dict(zip(wspd.dims, wspd.shape))}")
wspd

## 2 — Yearly mean wind speed with time slider

In [ ]:
%%time
wspd_yearly = wspd.resample(time="YE").mean().compute()

years = wspd_yearly.time.dt.year.values
wspd_yearly = (
    wspd_yearly
    .assign_coords(year=("time", years))
    .swap_dims({"time": "year"})
    .drop_vars("time")
)
wspd_yearly.attrs["long_name"] = "Annual Mean Wind Speed"
wspd_yearly.attrs["units"]     = "m/s"
print(f"Yearly wind speed shape: {dict(zip(wspd_yearly.dims, wspd_yearly.shape))}")
wspd_yearly

In [ ]:
wspd_yearly.hvplot(
    x="lon",
    y="lat",
    groupby="year",
    rasterize=True,
    geo=True,
    cmap="viridis",
    clim=(float(wspd_yearly.min()), float(wspd_yearly.max())),
    clabel="Wind Speed (m/s)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title="Mediterranean Annual Mean 10-m Wind Speed — use slider to step through years",
    fontscale=1.2,
    widget_type="scrubber",
    widget_location="bottom",
)

## 3 — Long-term climatological mean

Average every pixel over the full record. This is the reference for anomaly computation.

In [ ]:
%%time
wspd_clim = wspd.mean(dim="time").compute()
wspd_clim.attrs["long_name"] = f"Climatological Mean Wind Speed ({TIME_START[:4]}\u2013{TIME_END[:4]})"
wspd_clim.attrs["units"]     = "m/s"
print(f"Wind speed clim range: {float(wspd_clim.min()):.2f} \u2013 {float(wspd_clim.max()):.2f} m/s")
wspd_clim

In [ ]:
wspd_clim.hvplot(
    x="lon",
    y="lat",
    rasterize=True,
    geo=True,
    cmap="viridis",
    clabel="Wind Speed (m/s)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Climatological Mean 10-m Wind Speed ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)

## 4 — Monthly wind speed anomaly

Anomaly = monthly wind speed − pixel climatological mean.

`wspd_clim` (2-D) broadcasts over the time axis of `wspd` (3-D). The result stays lazy.

In [ ]:
wspd_anom = wspd - wspd_clim
wspd_anom.attrs["long_name"] = "Wind Speed Anomaly"
wspd_anom.attrs["units"]     = "m/s"
print(f"Anomaly shape: {dict(zip(wspd_anom.dims, wspd_anom.shape))}")
wspd_anom

## 4b — Monthly wind speed anomaly map with time slider

In [ ]:
%%time
wspd_anom_loaded = wspd_anom.compute()

anom_max = float(max(abs(wspd_anom_loaded.min()), abs(wspd_anom_loaded.max())))
times = wspd_anom_loaded.time.values

player = pn.widgets.Player(
    start=0, end=len(times) - 1, value=0,
    loop_policy="loop", width=800, name="",
)

def plot_wind_frame(idx):
    label = pd.Timestamp(times[idx]).strftime("%B %Y")
    return wspd_anom_loaded.isel(time=idx).hvplot(
        x="lon",
        y="lat",
        rasterize=True,
        geo=True,
        cmap="RdBu_r",
        clim=(-anom_max, anom_max),
        clabel="Wind Speed Anomaly (m/s)",
        tiles="OSM",
        frame_width=800,
        frame_height=450,
        title=f"Mediterranean Monthly Wind Speed Anomaly \u2014 {label}",
        fontscale=1.2,
    )

pn.Column(pn.bind(plot_wind_frame, player), player)

## 5 — Pixel-wise minimum and maximum anomaly

Both reductions are fused into a single dask graph pass.

In [ ]:
%%time
wspd_anom_min, wspd_anom_max = dask.compute(
    wspd_anom.min(dim="time"),
    wspd_anom.max(dim="time"),
)

wspd_anom_min.attrs["long_name"] = "Minimum Wind Speed Anomaly"
wspd_anom_min.attrs["units"]     = "m/s"
wspd_anom_max.attrs["long_name"] = "Maximum Wind Speed Anomaly"
wspd_anom_max.attrs["units"]     = "m/s"

print(f"Min anomaly range : {float(wspd_anom_min.min()):.3f} \u2013 {float(wspd_anom_min.max()):.3f} m/s")
print(f"Max anomaly range : {float(wspd_anom_max.min()):.3f} \u2013 {float(wspd_anom_max.max()):.3f} m/s")

## 6 — Spatial maps of min and max anomaly

In [ ]:
anom_abs_max = max(
    abs(float(wspd_anom_min.min())),
    abs(float(wspd_anom_max.max())),
)
clim_anom = (-anom_abs_max, anom_abs_max)
print(f"Symmetric color range: {clim_anom[0]:.3f} \u2013 {clim_anom[1]:.3f} m/s")

In [ ]:
plot_min = wspd_anom_min.hvplot(
    x="lon",
    y="lat",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Wind Speed Anomaly (m/s)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Minimum Wind Speed Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_min

In [ ]:
plot_max = wspd_anom_max.hvplot(
    x="lon",
    y="lat",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Wind Speed Anomaly (m/s)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Maximum Wind Speed Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_max

## 7 — Side-by-side comparison

In [ ]:
plot_min_small = wspd_anom_min.hvplot(
    x="lon", y="lat",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Wind Speed Anomaly (m/s)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Min Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

plot_max_small = wspd_anom_max.hvplot(
    x="lon", y="lat",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Wind Speed Anomaly (m/s)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Max Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

(plot_min_small + plot_max_small).cols(2)

## 8 — Annual wind speed anomaly map + MP4 export

Resample the monthly anomaly to annual means, animate one frame per year, and
save to `wind_anomaly_annual.mp4`.  Requires `ffmpeg` (available in this conda
environment via the `ffmpeg` package).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

VIDEO_PATH = "wind_anomaly_annual.mp4"

# Annual mean anomaly (wspd_anom_loaded is already in memory from section 4b)
wspd_anom_annual = wspd_anom_loaded.resample(time="YE").mean()
years_list = wspd_anom_annual.time.dt.year.values

lons = wspd_anom_annual.lon.values
lats = wspd_anom_annual.lat.values

# Symmetric colour limits across all years
ann_abs_max = float(max(
    abs(float(wspd_anom_annual.min())),
    abs(float(wspd_anom_annual.max())),
))

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor("#c8e6f5")

# First frame
data0 = wspd_anom_annual.isel(time=0).values
img = ax.pcolormesh(
    lons, lats, data0,
    cmap="RdBu_r",
    vmin=-ann_abs_max, vmax=ann_abs_max,
    shading="auto",
)
cb = fig.colorbar(img, ax=ax, label="Wind Speed Anomaly (m/s)", fraction=0.03, pad=0.02)
title = ax.set_title(
    f"Mediterranean Annual Wind Speed Anomaly — {years_list[0]}",
    fontsize=14,
)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.tight_layout()

def update(frame_idx):
    data = wspd_anom_annual.isel(time=frame_idx).values
    img.set_array(data.ravel())
    title.set_text(f"Mediterranean Annual Wind Speed Anomaly — {years_list[frame_idx]}")
    return img, title

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(years_list),
    interval=400,
    blit=False,
)

ani.save(VIDEO_PATH, writer="ffmpeg", fps=3, dpi=120)
plt.close(fig)
print(f"Video saved → {VIDEO_PATH}")

In [ ]:
from IPython.display import FileLink, display
display(FileLink(VIDEO_PATH, result_html_prefix="Download video: "))